# W02 — ML Task Framing
**Lane: Refresh / Content Opportunity Scoring (Lane 2)**

Mapping my lane onto the ML loop: task type, target, success metric, unit of analysis, and why ML earns its place here over a fixed rule.

*Read `skills/README.md`

## 1. My Lane as an ML Task (type)

**Task type: Classification, feeding a ranking/scoring output.**

Using the mapping from the framing skill: my question is "will this content item decline?" — that's a **classification** question (yes/no, from an observed outcome), not pure ranking and not clustering. But the classification's job isn't to output a label in isolation — its probability score is what gets sorted into the final **ranked review queue** (a scoring/ranking output), because the real decision is "which pages first," not "is this page declining, yes or no."

So concretely: a binary classifier predicts P(decline), and that probability is the ranking key for the review queue. This matches Lane 2's own good-methods list in the lane guide: logistic regression / decision tree / random forest / gradient boosting, evaluated with precision@K rather than plain accuracy — because the queue, not the raw label, is the actual product.

## 2. Target or Proxy

**Starter proxy target:** `is_declining_label = (trend_direction == "down")` — this is what the starter pipeline already uses. It's explicitly a **proxy**, not a real future outcome: `trend_direction` is itself computed from `trend_pct`, both derived from the *current* window, not a later one.

**Leakage rule I'm following (from the data skill):** because `trend_direction` and `trend_pct` are used to *build* the label, they can never also be used as *features* — otherwise the model would just be learning to reproduce its own answer (a circular result).

**Where this needs to go for the capstone:** the guide is explicit that a stronger version needs a genuine `prior 90 days of features -> next 30 days decline` design, built from the warehouse's daily fact table with a proper leakage audit. For this notebook (W02), I'm using the starter proxy to frame the task honestly, and flagging that the real target still needs to be built later, not treating the proxy as good enough long-term.

## 3. Success Metric

**Primary metric: Precision@K** (specifically Precision@50, matching the starter pipeline's own reporting) — because the output is a review queue a human works down in order, top-K metrics match how the list is actually used far better than plain accuracy or even overall AUC.

**Why not just accuracy:** with a review team that can only act on a fixed number of pages, what matters is "of the top 50 pages we send them, how many are actually worth reviewing" — not how the model does on the other 29,950 pages nobody will ever look at.

**Secondary metrics I'll also track:** ROC-AUC and average precision (for overall ranking quality), and recall on high-impression pages specifically — because per my W01 framing, a missed high-value declining page (false negative) costs more than a wasted review (false positive), so I care about not missing the big ones more than raw precision alone would tell me.

## 4. The Unit of Analysis, as a Real Dataframe

One row = **one content item's trailing-90-day performance snapshot** (`content_id` in the starter data). Loading the starter dataset and showing this concretely, plus a sketch of what the target column looks like.

In [14]:
import os

if not os.path.isdir("ml-internship"):
    !git clone https://github.com/m-husnain-dev/ml-internship.git

os.chdir("ml-internship")
print("Current directory:", os.getcwd())

Cloning into 'ml-internship'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 116 (delta 33), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 1.85 MiB | 15.38 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Current directory: /content/ml-internship/ml-internship/ml-internship/ml-internship/ml-internship


In [15]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
print("One row =", "one content_id's 90-day snapshot")
df[["content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days", "avg_position", "ctr", "trend_direction"]].head()

Rows: 30000 | Columns: 44
One row = one content_id's 90-day snapshot


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,44.0,0.13,down


**Sketching the target column** — building `is_declining_label` the same way the starter pipeline does, on the eligible slice (matching W01's filter: `impressions_90d > 0` and `content_age_days >= 90`):

In [16]:
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

print(f"Eligible rows: {len(eligible)}")
print(eligible["is_declining_label"].value_counts(normalize=True).rename("share"))
eligible[["content_id", "trend_direction", "is_declining_label", "impressions_90d", "avg_position", "ctr"]].head(10)

Eligible rows: 30000
is_declining_label
1    0.542067
0    0.457933
Name: share, dtype: float64


,content_id,trend_direction,is_declining_label,impressions_90d,avg_position,ctr
0,content_304f48230142,down,1,3803,10.6,0.76
1,content_a1fb4e703a9e,down,1,15320,20.3,0.05
2,content_9aa793d4d895,down,1,12581,36.5,0.09
3,content_331d6c4de07b,stable,0,11751,6.2,0.49
4,content_d99b7a2d90ca,down,1,19140,44.0,0.13
5,content_d4084a4bc775,down,1,3970,8.5,0.03
6,content_9a34b442b552,down,1,20,7.0,0.00
7,content_a63219c6e95a,stable,0,1724,21.2,0.06
8,content_5e6c160719bc,down,1,32574,46.0,0.09
9,content_c27558df2b0c,down,1,1240,4.9,0.16


## 5. Why ML Beats a Fixed Rule Here

A fixed rule (an if-statement baseline) can only combine a handful of thresholds explicitly — e.g. "flag if `days_since_last_update >= 180` AND `impressions_90d >= 500`." That's exactly what the starter's `baseline_refresh_score` does: a weighted sum of hand-picked rules.

The problem: real decline risk depends on **many weak, interacting signals at once** — position, CTR relative to position tier, freshness, word count, engagement rate, search volume, competition — and how they interact isn't obvious enough to hand-write correctly. A model can weigh dozens of these signals simultaneously and learn the interactions a human wouldn't think to encode.

**This isn't hypothetical for this lane — it's already measured in this repo.** The starter pipeline's own results (`outputs/model_report.md`):

| Method | Precision@50 |
|---|---|
| baseline rules | 0.240 |
| random forest | 0.740 |

The random forest catches roughly **3x more true positives in the same top-50 review capacity** than the hand-written rule, under proper client-holdout validation. That's real evidence a learned model earns its place here — not complexity added for its own sake.

## 6. Self-Check

- [x] Named the ML task type: classification (P(decline)) feeding a ranking/scoring output — not just "predict X."
- [x] Named the target/proxy (`is_declining_label`) and flagged it as a proxy, not a real future outcome, with the leakage rule stated (trend_direction/trend_pct: label only, never features).
- [x] Named the success metric (Precision@K, specifically @50) and explained why it matches the real decision better than plain accuracy.
- [x] Showed the unit of analysis as an actual dataframe — one row = one content item's 90-day snapshot — and sketched the target column on real data.
- [x] Explained why this is an ML problem and not just a fixed rule, backed by this repo's own measured baseline-vs-model gap (Precision@50: 0.240 → 0.740).
- [x] Tied the output to a real action: a ranked review queue a content reviewer works down, with limited capacity.